In [113]:
import pandas as pd
df_raw = pd.read_csv('../data/survey.csv')
df = df_raw.copy()
df.head()

,Record ID,Site,Cohort Year,DOB,Race,Please explain,Gender,Are you a U.S. Citizen or do you have a green card and permanent resident status?,"If not a U.S. Citizen or green card holder, are you an international student on a student visa or a DACA student?",Do you consider yourself coming from an economically or environmentally disadvantaged background?,...,7,8,9,10,11,12,13,14,15,16
0,4281-1,Moscow,2019-2021,34912.0,White.,NaN,Female,Yes,NaN,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1000001,Seattle/Olympia,2019-2021,34219.0,Asian,NaN,Male,NaN,NaN,Yes,...,,,,,,,,,,
2,1000002,Seattle/Olympia,2019-2021,29809.0,Black or African American,NaN,Male,NaN,NaN,Yes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1000003,Seattle/Olympia,2019-2021,34215.0,Native Hawaiian or Other Pacific Islander. <sp...,NaN,Female,NaN,NaN,Yes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1000005,Spokane,2019-2021,34417.0,Other,NaN,Female,NaN,NaN,No,...,,,,,,,,,,


In [114]:
# I want to combine two existing colums '
#"Are you a U.S. Citizen or do you have a green card and permanent resident status?" and "If not a U.S. Citizen or green card holder, are you an international student on a student visa or a DACA student? "
#into  one column, with 3 standardized answer options 1) green card/pr 2) daca or international student 3) other
# to do this, any "yes" answers in the first exist column map to answer 1. any no's in the first followed by a yes in the second map to standardized answer 2. otherwise, 3. 
#In the end, I want this new column to exist, and neither of the earlier two. what would the code for making these changes look like?

# Original column names (shortened for readability in the code)
col_citizen = "Are you a U.S. Citizen or do you have a green card and permanent resident status?"
col_visa = "If not a U.S. Citizen or green card holder, are you an international student on a student visa or a DACA student? "

def classify_citizenship(row):
    citizen_ans = str(row[col_citizen]).strip().lower()
    visa_ans = str(row[col_visa]).strip().lower()

    if citizen_ans == 'yes':
        return 'Green card/PR'
    elif (not (citizen_ans == 'yes')  )and( visa_ans == 'yes'):
        return 'DACA or International Student'
    elif (not (citizen_ans == 'yes')) and (visa_ans == 'no'): 
        return'other'
    else:
        return 'NaN'

# Create new column
### df['citizenship_status'] = df.apply(classify_citizenship, axis=1)

# Drop the original two columns
###df = df.drop(columns=[col_citizen, col_visa])

# Verify the result
#print(df['citizenship_status'].value_counts())

#df.head()

In [115]:
# you need to end up with 3 columns - 1) rural or underserved 2) rural 3) non-rural, underserved

#combining all the columns that ask them if they have every worked in a rural/underserved area into one
col_DL = "Have volunteered in a rural or underserved community" #answers are yes no or blank. can be in different cases
col_CV = "APR.STATUS" # a set of various long options, one or two related to underserved
col_AY = "Please select the option(s) that best describe your employment or further training. Select all " \
"that apply. (choice=I am currently employed or pursuing further training in a rural setting)" # some 'checked'. very few answers, not a column worth going over everytime
col_AW = "Please select the option(s) that best describe your employment or further training. Select all that apply." \
" (choice=I am currently employed or pursuing further training in a medically underserved community)" #same as AY
col_Y = "Have you ever worked or volunteered in a rural or underserved area doing general or health-related public service?" #Yes or No. very useful answers
col_howLong = "How long did you do this work or volunteering?" #AA
col_whereAndCapacity = "Where and in what capacity? " #Z ##qualitative answers


total = 0
worked = 0

def workedRuralOrUnderserved(row):
    DL_ans = str(row[col_DL]).strip().lower()
    Y_ans = str(row[col_Y]).strip().lower()
    CV_ans = str(row[col_CV]).strip().lower() 
    AY_ans = str(row[col_AY]).strip().lower()
    AW_ans = str(row[col_AW]).strip().lower()
    
    
    if ((DL_ans == 'yes') or (Y_ans == 'yes') or (AY_ans == 'checked')or (AW_ans == 'checked')):
        
        return ('Yes')
    elif(("underserved community" in CV_ans) or ("rural setting" in CV_ans)):
        return('Yes')
    
    elif ((DL_ans == 'no') and (Y_ans == 'no')):
        return('No')
    else:
        return('NaN')


#add function for zeros to how long?

    


    

In [117]:
## did they intend to work in a rural or underserved community?
col_AH = "INTENTIONS" # relevant options = 'Rural/unincorporated area', 'Urban Underserved' for sure 
# unsure about 'Small town (population less than 2,500)', 'Town (population 2,500 to 10,000-other than suburb)'
col_AV = "Do you intend to practice in a rural and/or medically underserved setting when your residency is completed?   " # look for 'yes'. only some answwers, but not necessarily useless
col_AT = "APR.INTENTIONS" # 'Individual intends to become employed or pursue further training in a medically underserved community' or 'Individual intends to become employed or pursue further training in a rural setting'


def intentions(row):
    AH_ans = str(row[col_AH]).strip().lower()
    AV_ans = str(row[col_AV]).strip().lower()
    AT_ans = str(row[col_AT]).strip().lower()

    if (AV_ans == 'yes'):
        return('Yes')
    elif (("underserved community" in AT_ans) or ("rural setting" in AT_ans)):
        return('Yes')
    elif ((AH_ans == 'rural/incorporated area') or (AH_ans == 'Urban Underserved') or (AH_ans == 'Small town (population less than 2,500)')):
        return('Yes') #NOT SURE IF THE OPTIONS FOR THIS NEED TO BE ADJUSTED
    else:
        return('No')
        

    
    


In [118]:
# do they come from a rural or underserved community?

col_T = "Please indicate the type of community you grew up in or consider your childhood home." # same answer options as AH
col_DJ = "From a non-rural underserved community?" #check for 'yes'
col_DK = "From a rural community" #check for 'yes'

def childhoodHome(row):
    T_ans = str(row[col_T]).strip().lower()
    DJ_ans = str(row[col_DJ]).strip().lower()
    DK_ans = str(row[col_DK]).strip().lower()
    
    if ((DK_ans =='yes') or (DJ_ans == 'yes')):
        return('Yes')
    elif((T_ans == 'rural/incorporated area') or (T_ans == 'Urban Underserved') or (T_ans == 'Small town (population less than 2,500)')):
        return('Yes') #ONCE AGAIN, OPTIONS MIGHT NEED TO BE ADJUSTED
    else:
        return('No')

In [119]:
# clean up the disadvantages stuff

#first combine some columns
col_DI = "Background - economically/environmentally disadvantaged background "
col_J = "Do you consider yourself coming from an economically or environmentally disadvantaged background?" ##!!

def combine_disadvantage1(row):
    DI_ans = str(row[col_DI]).strip().lower()
    J_ans = str(row[col_J]).strip().lower()

    if ((DI_ans == 'yes')or (J_ans == 'yes')):
        return 'Checked'

df['economically/environmentally_disadvantagedBackground'] = df.apply(combine_disadvantage1, axis=1)

col_K = "Are you the first person in your family to attend college?" #!!
col_N = "Disadvantaged status: please check as many as apply (choice=I am the first in my family to go to college)" #!!

def combine_disadvantage1(row):
    K_ans = str(row[col_K]).strip().lower()
    N_ans = str(row[col_N]).strip().lower()

    if ((K_ans == 'yes')or (N_ans == 'checked')):
        return 'Checked'
    
df['first_toAttend'] = df.apply(combine_disadvantage1, axis=1)

df = df.drop(columns = ["Background - economically/environmentally disadvantaged background ", "Do you consider yourself coming from an economically or environmentally disadvantaged background?", "Are you the first person in your family to attend college?", "Disadvantaged status: please check as many as apply (choice=I am the first in my family to go to college)", "Disadvantaged status: please check as many as apply (choice=None of the above)" ])



##now for putting them all in one column 

col_list = [
    "economically/environmentally_disadvantagedBackground",
    "first_toAttend", 
    "Disadvantaged status: please check as many as apply (choice=I grew up with English as my second language)", 
    "Disadvantaged status: please check as many as apply (choice=I have been diagnosed with a physical or mental impairment that limits my participation)", 
    "Disadvantaged status: please check as many as apply (choice=I qualify for federal/state grants which do not need to be repaid)", 
]

col_labels = {
    "economically/environmentally_disadvantagedBackground" : "Comes from an economically or environmentally disadvantaged background", 
    "first_toAttend" : "Is the first in their family to attend college", 
    "Disadvantaged status: please check as many as apply (choice=I grew up with English as my second language)" : "Has English as a second language", 
    "Disadvantaged status: please check as many as apply (choice=I have been diagnosed with a physical or mental impairment that limits my participation)" : "Has a diagnosed physical or mental impairment", 
    "Disadvantaged status: please check as many as apply (choice=I qualify for federal/state grants which do not need to be repaid)" : "Qualified for Federal/State grants that do not need to be repaid" 
}

df['Disadvantaged Status'] = df.apply(
    lambda row: ", ".join(
        col_labels[col]
        for col in col_list
        if str(row[col]).strip().lower() == "checked"
    ),
    axis=1
)

df['Disadvantaged Status'] = df['Disadvantaged Status'].replace("", "None Apply")

df = df.drop(columns = ["economically/environmentally_disadvantagedBackground", "first_toAttend", "Disadvantaged status: please check as many as apply (choice=I grew up with English as my second language)", "Disadvantaged status: please check as many as apply (choice=I have been diagnosed with a physical or mental impairment that limits my participation)", "Disadvantaged status: please check as many as apply (choice=I qualify for federal/state grants which do not need to be repaid)"])





In [120]:
df = df.drop(columns = ["Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am participating in a residency program)", "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=None of the above)"])
col_list2 = [
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am currently employed or pursuing further training in a primary care setting)",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am employed at a National Health Service Corp approved site)",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am providing behavioral health services)",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am providing maternal health care)",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am serving individuals with OUD/SUD)"
]

col_labels2 = {
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am currently employed or pursuing further training in a primary care setting)": "Primary Health Care Setting",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am employed at a National Health Service Corp approved site)": "National Health Service Corp approved site",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am providing behavioral health services)": "Behavioral Health Services",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am providing maternal health care)": "Maternal Health Care",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am serving individuals with OUD/SUD)" :"Serving Individuals with OUD/SUD"
}

df['Describe your employment or further training setting'] = df.apply(
    lambda row: ", ".join(
        col_labels2[col]
        for col in col_list2
        if str(row[col]).strip().lower() == "checked"
    ),
    axis=1
)

#df['Describe your employment or further training setting'] = df['Describe your employment or further training setting'].replace("", "None Apply")

df = df.drop(columns = ["Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am currently employed or pursuing further training in a primary care setting)",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am employed at a National Health Service Corp approved site)",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am providing behavioral health services)",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am providing maternal health care)",
    "Please select the option(s) that best describe your employment or further training. Select all that apply. (choice=I am serving individuals with OUD/SUD)"])




In [85]:
for col in df.columns:
    if "Interprofessional collaboration" in col:
        print(repr(col))

'5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Interprofessional collaboration)'


In [121]:

#ADD Diferentiation between no responses vs answering N/A?



col_list3 = [
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Interprofessional collaboration)", 
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Behavioral health integration)", 
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Social determinants of health)",
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Cultural competency)",
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Practice transformation)", 
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Current and emerging health issues)"
    
]
col_labels3 = {
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Interprofessional collaboration)": "Interprofessional collaboration", 
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Behavioral health integration)": "Behavioral health Integration", 
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Social determinants of health)": "Social determinants of health",
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Cultural competency)" : "Cultural competency",
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Practice transformation)": "Practice transformation", 
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Current and emerging health issues)": "Current and emerging health issues" 
    
}

df['Describe your employment or further training setting'] = df.apply(
    lambda row: ", ".join(
        col_labels3[col]
        for col in col_list3
        if str(row[col]).strip().lower() == "checked"
    ),
    axis=1
)

df = df.drop(columns = ["5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Interprofessional collaboration)", 
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Behavioral health integration)", 
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Social determinants of health)",
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Cultural competency)",
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Practice transformation)", 
    "5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=Current and emerging health issues)"
    ])

df = df.drop(columns = ["5.\tSelect the topics addressed/incorporated in your current work/professional setting. Select all that apply.  (choice=N/A)"])


In [122]:
# columns that I just want to drop
col_F = "Please explain" # please explain
col_CM = "APR.No IDENTIFIER" # all blank. idk what it's doing
col_CO = "APR.Record ID"
col_CP = "APR.GENDER"
col_CQ = "APR.RACE"

col_DB = "Record ID.1"
col_DE = "Site.1"
col_DF = "Cohort Year.1"
col_DG = "Gender.1"
col_DH = "Race.1"

col_CJ = "Column88"
col_CK = "Column89"

df = df.drop(columns = [col_F, col_CM, col_CO, col_CP, col_CQ, col_CJ, col_CK, col_DB, col_DE, col_DF, col_DG, col_DH])



In [123]:
#final 

df['citizenship_status'] = df.apply(classify_citizenship, axis=1)
df['worked_rural_orUnderserved'] = df.apply(workedRuralOrUnderserved, axis=1)
df['intended_rural_orUnderserved'] = df.apply(intentions, axis=1)
df['cameFrom_rural_orUnderserved'] = df.apply(childhoodHome, axis=1)






df = df.drop(columns=[col_citizen, col_visa])
df = df.drop(columns=[col_DL, col_Y,  col_AW, col_AY])
df = df.drop(columns = [col_AV])
df = df.drop(columns = [col_DJ, col_DK]) 
#SHOULD I DROP T and AH or is that information useful, still?
df = df.drop(columns = ["Are you from a non-rural underserved community*?    *Urban community that includes members of minority populations or individuals who have experienced health disparities.", "Where? What community? ", "Please explain2", "Specify other", "Signature", "Column3", "Date", "Planned setting of occupation after program", "APR.STATUS", "APR.INTENTIONS", "APR.DISADV", "APR.RURAL", "APR.DOB", "If you answered 'yes', please complete question 2b. below. If not, please continue to question 3.   2b. Select the barriers you experienced in finding a position in the medically underserved community, rural community, and/or primary care? Select all that",
"If you answered 'yes', please complete question 2b. below. If not, please continue to question 3.   2b. Select the barriers you experienced in finding a position in the medically underserved community, rural community, and/or primary care? Select all t2",
"If you answered 'yes', please complete question 2b. below. If not, please continue to question 3.   2b. Select the barriers you experienced in finding a position in the medically underserved community, rural community, and/or primary care? Select all t3",
"If you answered 'yes', please complete question 2b. below. If not, please continue to question 3.   2b. Select the barriers you experienced in finding a position in the medically underserved community, rural community, and/or primary care? Select all t4",
"If you answered 'yes', please complete question 2b. below. If not, please continue to question 3.   2b. Select the barriers you experienced in finding a position in the medically underserved community, rural community, and/or primary care? Select all t5",
"If you answered 'yes', please complete question 2b. below. If not, please continue to question 3.   2b. Select the barriers you experienced in finding a position in the medically underserved community, rural community, and/or primary care? Select all t6",
"If you answered 'yes', please complete question 2b. below. If not, please continue to question 3.   2b. Select the barriers you experienced in finding a position in the medically underserved community, rural community, and/or primary care? Select all t7",
"If you answered 'yes', please complete question 2b. below. If not, please continue to question 3.   2b. Select the barriers you experienced in finding a position in the medically underserved community, rural community, and/or primary care? Select all t8",
"If you answered 'yes', please complete question 2b. below. If not, please continue to question 3.   2b. Select the barriers you experienced in finding a position in the medically underserved community, rural community, and/or primary care? Select all t9",
"If you answered 'yes', please complete question 2b. below. If not, please continue to question 3.   2b. Select the barriers you experienced in finding a position in the medically underserved community, rural community, and/or primary care? Select all t10",
"If you answered 'yes' to 2a, but none of the choices in 2b apply, please explain your answer for 'Other' here . ", '\xa0',
'\xa02',
'\xa03',
'\xa04',
'\xa05',
'\xa06',
'\xa07',
'\xa08',
'\xa09',
'\xa010',
'\xa011',
'\xa012',
'\xa013',
'\xa014',
'\xa015',
'\xa016'])

df.head()






,Record ID,Site,Cohort Year,DOB,Race,Gender,"Does your father, mother, or spouse (or the previous employment, if no longer working or deceased) work in healthcare?",Who?,Veteran Status,Please indicate the type of community you grew up in or consider your childhood home.,...,HPSA - Dental Care,HPSA - Mental Health,RUCA Code,FQHC,Disadvantaged Status,Describe your employment or further training setting,citizenship_status,worked_rural_orUnderserved,intended_rural_orUnderserved,cameFrom_rural_orUnderserved
0,4281-1,Moscow,2019-2021,34912.0,White.,Female,No,NaN,Not a veteran,Rural/unincorporated area,...,yes,yes,1,no,Is the first in their family to attend college,,Green card/PR,Yes,Yes,Yes
1,1000001,Seattle/Olympia,2019-2021,34219.0,Asian,Male,No,NaN,Not a veteran,"Small city (population 10,000 to 50,000-other ...",...,no,yes,1,no,Comes from an economically or environmentally ...,,NaN,Yes,Yes,No
2,1000002,Seattle/Olympia,2019-2021,29809.0,Black or African American,Male,No,NaN,Not a veteran,Suburb of a large city,...,NaN,NaN,NaN,NaN,Comes from an economically or environmentally ...,,NaN,Yes,Yes,No
3,1000003,Seattle/Olympia,2019-2021,34215.0,Native Hawaiian or Other Pacific Islander. <sp...,Female,Yes,Mother and father,Not a veteran,"Town (population 2,500 to 10,000-other than su...",...,NaN,NaN,NaN,NaN,Comes from an economically or environmentally ...,,NaN,Yes,No,No
4,1000005,Spokane,2019-2021,34417.0,Other,Female,No,NaN,Not a veteran,Rural/unincorporated area,...,yes,yes,1,no,None Apply,,NaN,Yes,No,Yes


In [124]:
for col in df.columns:
    
    print(repr(col))

'Record ID'
'Site'
'Cohort Year'
'DOB'
'Race'
'Gender'
'Does your father, mother, or spouse (or the previous employment, if no longer working or deceased) work in healthcare?'
'Who?'
'Veteran Status'
'Please indicate the type of community you grew up in or consider your childhood home.'
'Where do you live now?'
'Where and in what capacity? '
'How long did you do this work or volunteering?'
'What is your school or university?'
'What is the name of your program or major?'
'Discipline'
'What degree will you be earning? '
'What year of your program are you in?'
'When (what month and year) do you anticipate graduating from your primary program? '
'INTENTIONS'
'In 250-500 words, please write a brief explanation about why you are interested in being a WA AHEC Scholar.'
'Please select the option that best describes your current employment/enrollment status. '
'Please select the option that best describes your current employment/enrollment status. _1'
'Name'
'Street Address'
'City'
'State'
'Zip

In [125]:
df.to_excel('cleaned_survey.xlsx', index=False)

In [ ]:

#combining identification (such as columns for race, gender, etc)